# Notebook 05 — SqueezeNet 1.1 on ImageNet (pretrained)

Uses a **pretrained SqueezeNet 1.1** (1.24 M params, 58.2 % Top-1, no residual connections, no FC layers) for BFT analysis. Weights download automatically from torchvision (~4.7 MB).

**Why SqueezeNet?** Smallest pretrained ImageNet model with no residual connections, and no large FC layers — the classifier is a single `Conv2d(512, 1000, 1)`, whose joint arbor matrix is only ~2 GB vs 75+ GB for AlexNet's first FC layer.

**Architecture note — squeeze spine:** SqueezeNet Fire modules have *parallel* expand branches, so BFT traces only the sequential spine (initial conv, the 8 squeeze convs, classifier conv).

## Structure
0. **Imports & setup**
1. **Configuration**
2. **Backward Factor Trace** — model, BFT, exploratory plots (factor panels, input factors, spatial maps)
3. **BFT figures** — main paper and appendix (placeholders)
4. **Fingerprints** — NNLS round-trip, ID sanity check, far-OOD, embeddings
5. **Fingerprint figures** — main paper and appendix (placeholders)

Validation, robustness and ablation analyses live in notebook 09.

## §0 — Imports & setup

In [ ]:
#%matplotlib inline
import sys, os
sys.path.insert(0, '..')

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
from torchvision.models import squeezenet1_1, SqueezeNet1_1_Weights
from sklearn.metrics.pairwise import paired_cosine_distances
from torch.utils.data import DataLoader, Subset, TensorDataset

from src import (
    collect_layer_dicts, bft, extract_tree_nodes, plot_factor_tree,
    extract_fingerprint_matrix, compute_stimulus_similarity, project_stimuli_onto_tree,
    project_onto_bft, extract_factor_tree_nodes, compute_factor_activations, nodes_at_layer,
    plot_factor_overview_panel, plot_factor_gallery, plot_input_layer_factors,
    plot_embedding_comparison, plot_spatial_activation_maps, plot_similarity_heatmap,
    imdenorm as _imdenorm,
)

plt.rcParams.update({'figure.dpi': 80})
DEVICE = ('cuda' if torch.cuda.is_available() else
          'mps'  if torch.backends.mps.is_available() else 'cpu')
print('device:', DEVICE)

## §1 — Configuration

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
IMAGENET_DIR = '../data'
FIG_DIR      = '../figs/05_imagenet_squeezenet'
os.makedirs(FIG_DIR, exist_ok=True)

# ── ImageNet constants ────────────────────────────────────────────────────────
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)
N_CLASSES     = 1000
IMG_SIZE      = 224

# ── Focus categories ──────────────────────────────────────────────────────────
N_SAMPLES_PER_CATEGORY = 250   # C0 scale-up (was 100; val-limited ~544)

CATEGORY_CLASSES = {
    'airplane': [404, 895],
    'ship':     [403, 724],
    'car':      [609, 751],
    'bicycle':  [444, 671],
    'elephant': [101, 385],
    'bear':     [294, 297],
    'dog':      [151, 251],
    'bird':     [7, 9],
}

CATEGORY_NAMES = list(CATEGORY_CLASSES.keys())
N_FOCUS        = len(CATEGORY_NAMES)
ALL_FOCUS_IDX  = sorted(set(i for idxs in CATEGORY_CLASSES.values() for i in idxs))
IDX_TO_CAT     = {idx: ci for ci, idxs in enumerate(CATEGORY_CLASSES.values()) for idx in idxs}

# Dict class_names for plot utilities
CAT_CLASS_NAMES = {i: CATEGORY_NAMES[i] for i in range(N_FOCUS)}

# ── BFT hyperparameters ───────────────────────────────────────────────────────
# Selected by the nb09 S10 grid, completed and decided by notebook 10
# (logs/results/nb10_imagenet_cnn.json). See PUBLICATION_SETTINGS.md.
# K_MAX is the "rank x1.3" profile — round(1.3 x K) on the rank each layer
# auto-selected under the old [4]*9 + [8]. Against that incumbent: silhouette
# 0.362 -> 0.411, 5-NN 0.940 -> 0.950, NMF stability 0.844 -> 0.886, fingerprint
# 223 -> 234 dims. The gap is far outside the noise: paired 80 % stimulus
# subsampling (B = 1000) puts the difference at +0.049 [0.044, 0.054], and x1.3
# beat the incumbent in 1000/1000 subsamples.
# NOTE the "rank x0.7" rule that won on both MLPs and the ViT is LAST here
# (silhouette 0.294) — on conv spines rank wants to go up, not down. x1.3 is also
# the TOP of the probed range, so the ImageNet optimum may be higher still.
# NOTE the trailing 10 is a rank, not a category count — it is no longer N_FOCUS.
# The three 2s were emitted by the sweep as 1s; _auto_k_factorize floors k_max at
# 2 (src/bft.py), so this profile is what nb10 actually executed.
K_MAX          = [6, 5, 7, 6, 7, 6, 6, 4, 6, 16]   # C0 circuit tree (nb15)
N_BRANCHES     = [1, 1, 1, 1, 1, 1, 1, 1, 2, 16]   # C0 circuit tree (nb15)
POOL_METHOD    = 'avg'                            # inherited, never swept
STIM_THRESHOLD = 0.0

print('Config ready.')
print(f'Categories ({N_FOCUS}): {CATEGORY_NAMES}')
print(f'Total focus ImageNet classes: {len(ALL_FOCUS_IDX)}')

# ── Fingerprint-tree HPs (paper/submission settings) — §4/§5 only ─────────────
K_MAX_FP     = [4, 4, 4, 4, 4, 4, 4, 4, 4, 8]
N_BRANCHES_FP = [1, 1, 1, 1, 1, 1, 1, 1, 2, 5]


## §2 — Backward Factor Trace

Layer data is collected via the **squeeze spine filter** — only the initial conv,
the 8 squeeze convs inside each Fire module, and the final classifier conv are captured.
This preserves BFT's sequential-layer assumption despite SqueezeNet's parallel expand branches.

### 2a — Data, model and layer activations

In [3]:
# ── Data loaders ──────────────────────────────────────────────────────────────
normalize = T.Normalize(IMAGENET_MEAN, IMAGENET_STD)
test_tfm  = T.Compose([
    T.Resize(256),
    T.CenterCrop(IMG_SIZE),
    T.ToTensor(),
    normalize,
])

def _make_imagenet(split, transform):
    """Load ImageNet; falls back to ImageFolder if ILSVRC structure absent."""
    try:
        return torchvision.datasets.ImageNet(IMAGENET_DIR, split=split, transform=transform)
    except Exception:
        folder = 'train' if split == 'train' else 'val'
        return torchvision.datasets.ImageFolder(
            os.path.join(IMAGENET_DIR, folder), transform=transform)

val_ds   = _make_imagenet('val', test_tfm)
val_loader = DataLoader(val_ds, batch_size=256, shuffle=False, num_workers=4, pin_memory=True)

CLASS_NAMES = val_ds.classes  # synset IDs, e.g. 'n01440764'
print(f'Val: {len(val_ds):,}   Classes: {N_CLASSES}')
print(f'Focus class synsets: {[CLASS_NAMES[c] for c in ALL_FOCUS_IDX]}')

Val: 50,000   Classes: 1000
Focus class synsets: ['n01514668', 'n01514859', 'n01871265', 'n02085620', 'n02085782', 'n02132136', 'n02133161', 'n02504013', 'n02687172', 'n02690373', 'n02701002', 'n02814533', 'n02835271', 'n02951358', 'n03792782', 'n04552348']


/home/jb3879/Factor_Trace/.venv/lib64/python3.11/site-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 3, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [ ]:
# ── Focused val loader (focus categories only) ────────────────────────────────
val_targets  = np.array(val_ds.targets)
focus_idx    = np.where(np.isin(val_targets, ALL_FOCUS_IDX))[0]
focus_val_ds = Subset(val_ds, focus_idx)
focus_loader = DataLoader(focus_val_ds, batch_size=256, shuffle=False, num_workers=4)
print(f'Focus val samples: {len(focus_val_ds)}')

In [ ]:
# ── Load pretrained SqueezeNet 1.1 ────────────────────────────────────────────
model = squeezenet1_1(weights=SqueezeNet1_1_Weights.IMAGENET1K_V1).to(DEVICE)
model.eval()
n_params = sum(p.numel() for p in model.parameters())
print(f'SqueezeNet 1.1 — parameters: {n_params:,}')

# Per-category top-1 accuracy on focus val samples
cat_correct = np.zeros(N_FOCUS)
cat_total   = np.zeros(N_FOCUS)
with torch.no_grad():
    for x, y in focus_loader:
        x = x.to(DEVICE)
        preds    = model(x).argmax(1).cpu().numpy()
        yt       = y.numpy()
        true_cats = np.array([IDX_TO_CAT.get(int(t), -1) for t in yt])
        pred_cats = np.array([IDX_TO_CAT.get(int(p), -1) for p in preds])
        for ci in range(N_FOCUS):
            mask = true_cats == ci
            cat_correct[ci] += (pred_cats[mask] == ci).sum()
            cat_total[ci]   += mask.sum()

cat_acc = cat_correct / (cat_total + 1e-12)
for ci, name in enumerate(CATEGORY_NAMES):
    print(f'  {name:10s}  Top-1: {cat_acc[ci]:.3f}  ({int(cat_correct[ci])}/{int(cat_total[ci])})')
print(f'\nMean category Top-1: {cat_acc.mean():.3f}')
print('(Full val set: Top-1 ≈ 0.582, Top-5 ≈ 0.806 per torchvision)')

In [ ]:
imdenorm = lambda img: _imdenorm(img, IMAGENET_MEAN, IMAGENET_STD)

def squeezenet_spine_filter(name, mod):
    """Select the sequential squeeze-spine: initial conv, squeeze convs, classifier conv."""
    return (name == 'features.0' or
            name == 'classifier.1' or
            (isinstance(mod, nn.Conv2d) and name.endswith('.squeeze')))

def filter_by_category(raw, n_per_category,
                        idx_to_cat=IDX_TO_CAT, n_categories=N_FOCUS):
    """Map ImageNet indices → category labels (0–N_FOCUS-1), sample n_per_category each.

    Samples are sorted by model confidence (highest first) within each category.
    Returns (filtered_dict, keep_indices).
    """
    orig_targets = raw['targets']
    cat_targets  = np.array([idx_to_cat.get(int(t), -1) for t in orig_targets])
    keep = []
    for ci in range(n_categories):
        ci_idx = np.where(cat_targets == ci)[0]
        if 'confidences' in raw and len(ci_idx):
            ci_idx = ci_idx[np.argsort(raw['confidences'][ci_idx])[::-1]]
        if len(ci_idx) > n_per_category:
            ci_idx = ci_idx[:n_per_category]
        keep.append(ci_idx)
    keep = np.sort(np.concatenate(keep))
    layer_data = [{**ld, 'input_fmap':  ld['input_fmap'][keep],
                         'output_fmap': ld['output_fmap'][keep]}
                  for ld in raw['layer_data']]
    result = {'images':       raw['images'][keep],
              'targets':      cat_targets[keep],      # category labels 0–7
              'orig_targets': orig_targets[keep],     # original ImageNet indices
              'layer_data':   layer_data}
    if 'confidences' in raw:
        result['confidences'] = raw['confidences'][keep]
    return result, keep


print('Collecting focus-category val data (squeeze spine only) …')
raw0 = collect_layer_dicts(model, focus_loader, DEVICE,
                            only_correct=True,
                            layer_filter=squeezenet_spine_filter)


data0, _ = filter_by_category(raw0, N_SAMPLES_PER_CATEGORY)
all_images0   = data0['images']
all_targets0  = data0['targets']      # category labels 0–7
layer_inputs0 = [ld['input_fmap'] for ld in data0['layer_data']]
n_samples0    = len(all_images0)

print(f'{n_samples0} samples | {len(layer_inputs0)} spine layers')
print(f'Layer shapes: {[x.shape for x in layer_inputs0]}')
print(f'Layer names: {[ld["name"] for ld in data0["layer_data"]]}')
print(f'Category counts: {dict(zip(CATEGORY_NAMES, [int((all_targets0==i).sum()) for i in range(N_FOCUS)]))}')


### 2b — Run BFT

In [ ]:
# ── Run BFT ───────────────────────────────────────────────────────────────────

print('Running BFT …')
from src import cached_tree
tree_root0 = cached_tree('nb05_circuit', lambda: bft(
    data0['layer_data'],
    k_max=K_MAX, n_branches=N_BRANCHES,
    conv_pool_method=POOL_METHOD,
    stimulus_threshold=STIM_THRESHOLD,
    weighting='img_selectivity', verbose=1, n_jobs=3,
), params=dict(k=K_MAX, b=N_BRANCHES, tau=STIM_THRESHOLD, n=len(all_targets0)))

tree_nodes0   = extract_tree_nodes(tree_root0)
factor_nodes0 = extract_factor_tree_nodes(tree_root0)
l0_nodes0     = nodes_at_layer(tree_root0, 0)
K_root        = len(tree_root0.root.lambdas)
print(f'Tree nodes: {len(tree_nodes0)}   K_root: {K_root}   L0 leaves: {len(l0_nodes0)}')


### 2c — Exploratory plots: factor overview panels and galleries

In [ ]:
# ── Plot 1+4: Factor overview panels & per-factor galleries (all tree nodes) ──
for node in tree_nodes0:
    layer_name = node.get('layer_name', f'L{node["layer_idx"]}')
    path_label = 'F' + '-F'.join(str(f) for f in node['path']) if node['path'] else 'root'
    node_id = f"{layer_name}_{path_label}"
    figs1 = plot_factor_overview_panel(node, all_images0, all_targets0, CAT_CLASS_NAMES)
    for k, fig in enumerate(figs1):
        fig.savefig(os.path.join(FIG_DIR, f'factor_overview_{node_id}_k{k}.pdf'),
                    bbox_inches='tight')
        plt.close(fig)
    K = node['img_factors'].shape[1]
    for k in range(K):
        fig4 = plot_factor_gallery(node, all_images0, all_targets0, CAT_CLASS_NAMES, k=k, n=10)
        fig4.savefig(os.path.join(FIG_DIR, f'factor_gallery_{node_id}_k{k}.pdf'),
                     bbox_inches='tight')
        plt.close(fig4)

print(f'Plot 1+4 saved for {len(tree_nodes0)} tree nodes.')

### 2d — Exploratory plots: input-layer spatial factors

In [ ]:
# ── Plot 2: Input-layer spatial factors (conv_rgb for initial conv, bars for squeeze) ──
for leaf in l0_nodes0:
    layer_name = leaf.layer_name
    path_label = 'F' + '-F'.join(str(f) for f in leaf.path) if leaf.path else 'root'
    node_id = f"{layer_name}_{path_label}"
    ld = data0['layer_data'][leaf.layer_idx]
    C_in = ld['weight'].shape[1]
    arch = 'conv_rgb' if C_in == 3 else 'fc'  # initial conv is RGB; squeeze convs are 1×1
    figs2 = plot_input_layer_factors(leaf, all_images0, arch=arch,
                                      image_shape=(3, IMG_SIZE, IMG_SIZE))
    for k, fig in enumerate(figs2):
        fig.savefig(os.path.join(FIG_DIR, f'input_factors_{node_id}_k{k}.pdf'),
                    bbox_inches='tight')
        plt.close(fig)

print(f'Plot 2 saved for {len(l0_nodes0)} L0 leaf nodes.')


### 2e — Exploratory plots: spatial activation maps

In [ ]:
# ── Spatial activation maps (only for initial conv — it has spatial extent) ───
spatial_leaves = [n for n in l0_nodes0 if n.layer_name == 'features.0']
for leaf in spatial_leaves[:2]:
    path_label = 'F' + '-F'.join(str(f) for f in leaf.path) if leaf.path else 'root'
    node_id = f"{leaf.layer_name}_{path_label}"
    fig = plot_spatial_activation_maps(model, all_images0, leaf, data0['layer_data'], DEVICE,
                                        n_images=6, denorm_fn=imdenorm,
                                        title=f'{node_id}: spatial activation maps')
    plt.savefig(os.path.join(FIG_DIR, f'spatial_{node_id}.pdf'), bbox_inches='tight')
    plt.show()


## §3 — BFT figures (main paper & appendix)

### 3a — Main-paper figure

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# EXPORT — figure data for the SqueezeNet / ImageNet circuits
# Requires: tree_root0, model, all_images0, all_targets0  (§2 above)
# Writes:   figures/figdata/nb05_circuits.npz  (+ .json)
# The figures are built in notebooks/fig05_imagenet.ipynb from this bundle alone.
# Tighter size guards than the small models: 224px stimuli and 512x256 channel
# matrices do not belong in a committed bundle, so images are downsampled and
# oversized connection maps are stored as marginals.
# ═══════════════════════════════════════════════════════════════════════════════
from src import (figdata, figexport, build_scaffold_edges,
                 scaffold_loading_from_edges,
                 scaffold_layer_sizes_from_edges)

root = tree_root0.root
CLASSES = [int(c) for c in np.unique(all_targets0)]


def spatial_maps(node, n_images=4):
    """Top stimuli of factor 0 and their channel-weighted activation maps."""
    C_out = node.weight.shape[0]
    ch = np.maximum(node.connection_factors[:, 0].reshape(C_out, -1).sum(1), 0)
    ch = ch / (ch.sum() + 1e-12)
    idx = np.argsort(node.img_factors[:, 0])[::-1][:n_images]
    store = {}
    mod = dict(model.named_modules())[node.layer_name]
    h = mod.register_forward_hook(lambda m, i, o: store.update(out=o.detach().cpu()))
    with torch.no_grad():
        model(torch.from_numpy(all_images0[idx]).float().to(DEVICE))
    h.remove()
    fm = store['out'].numpy()
    return dict(index=idx, images=all_images0[idx][:, :, ::4, ::4].astype(np.float32),
                maps=np.maximum((fm * ch[None, :, None, None]).sum(1), 0))


circuits = []
for child in root.children:
    chain, leaf = [root, child], child
    while leaf.children:
        leaf = leaf.children[0]
        chain.append(leaf)
    fwd = list(reversed(chain))                                   # input-first
    E, negE = build_scaffold_edges(fwd[1:], fi='path', fi_seed=fwd[0], top_pct=0.05)
    prof = figexport.class_profile(root, all_targets0, CLASSES)[int(child.path[0])]
    circuits.append(dict(
        k=int(child.path[0]), profile=prof,
        top_classes=[int(CLASSES[c]) for c in np.argsort(prof)[::-1] if prof[c] >= 0.15],
        leaf_layer=str(leaf.layer_name), depth=len(chain),
        scaffold=figexport.scaffold_summary(
            E, negE, scaffold_loading_from_edges(E),
            scaffold_layer_sizes_from_edges(E))))

_stim = figexport.subsample_by_class(all_targets0, CLASSES, 60, seed=0)
nodes = figexport.export_tree(root, labels=all_targets0, classes=CLASSES,
                              images=all_images0, stim_idx=_stim, n_top=5,
                              max_matrix=150_000, img_max_side=48,
                              example_max_side=64)
_bfs, _q = [], [root]
while _q:
    _n = _q.pop(0); _bfs.append(_n); _q.extend(_n.children)
for _d, _n in zip(nodes, _bfs):
    if _n.layer_type == 'conv':
        try:
            _d['spatial'] = spatial_maps(_n)
        except Exception as _e:                     # never lose the bundle over a hook
            print(f'  spatial maps skipped for {_n.layer_name}: {_e}')

D = figdata.save('nb05_circuits', dict(
    classes=CLASSES, category_names=[str(c) for c in CATEGORY_NAMES],
    class_names=[str(CAT_CLASS_NAMES[c]) if c in CAT_CLASS_NAMES else str(c)
                 for c in CLASSES],
    image_mean=list(IMAGENET_MEAN), image_std=list(IMAGENET_STD), img_size=IMG_SIZE,
    meta=figexport.trace_meta(tree_root0, k_max=K_MAX, n_branches=N_BRANCHES,
                              pool_method=POOL_METHOD,
                              stimulus_threshold=STIM_THRESHOLD),
    circuits=circuits, nodes=nodes,
    stimuli=figexport.example_stimuli(all_images0, all_targets0, CLASSES,
                                      per_class=4, max_side=64)))
figdata.summary('nb05_circuits')


### 3b — Appendix figure

In [ ]:
# (appendix figure: build it in notebooks/fig05_imagenet.ipynb from the exported bundle)


## §4 — Fingerprints

In [ ]:
# ── Two trees: circuits use the C0 tree (above); fingerprints use a shallower
#    paper-HP tree (nb16 C1.8). Both cached.
from src import cached_tree
tree_circuit = tree_root0                    # keep the C0 circuit tree for §7/§8
tree_root0 = cached_tree('nb05_fp', lambda: bft(
    data0['layer_data'], k_max=K_MAX_FP, n_branches=N_BRANCHES_FP,
    conv_pool_method=POOL_METHOD, stimulus_threshold=STIM_THRESHOLD, weighting='img_selectivity', n_jobs=3),
    params=dict(k=K_MAX_FP, b=N_BRANCHES_FP, tau=STIM_THRESHOLD, n=len(all_targets0)))
print('circuit tree:', sum(1 for _ in tree_circuit.nodes()), 'nodes | '
      'fingerprint tree:', sum(1 for _ in tree_root0.nodes()), 'nodes')

ANALYSIS_CTX = dict(
    tag='nb05', model=model, tree_circuit=tree_circuit, tree_fp=tree_root0,
    layer_inputs=[d['input_fmap'] for d in data0['layer_data']], labels_task=all_targets0.astype(int),
    labels_fine=all_targets0.astype(int), eval_loader=val_loader,
    label_transform=None, device=DEVICE,
    layer_names=[d['name'] for d in data0['layer_data']],
    n_classes=8, k_cap=16, last_extra=4, k_max_cfg=K_MAX,
    skip_pruning='ImageNet pruning is the heavy last step — run via nb14 / EXPERIMENTS_TO_RUN Phase 4',
    prune_fractions=(0.02, 0.05, 0.1, 0.2), n_random=5, stab_seeds=5)

In [ ]:
# §3 only exports plot data — the paper figures live in
# notebooks/fig05_imagenet.ipynb — so nothing above changed matplotlib's rcParams.
# Kept so the exploratory plots below render at screen size.
plt.rcParams.update({'figure.dpi': 80})


### 4a — NNLS round-trip fidelity

In [ ]:
# ── Round-trip test ───────────────────────────────────────────────────────────
N_RT   = min(200, n_samples0)
rng_rt = np.random.default_rng(0)
rt_sub = rng_rt.choice(n_samples0, N_RT, replace=False)
rt_inputs = [l[rt_sub] for l in layer_inputs0]

projected_rt = project_stimuli_onto_tree(tree_root0, rt_inputs)
F_orig_rt    = extract_fingerprint_matrix(tree_root0, rt_sub)
F_rt         = extract_fingerprint_matrix(projected_rt, np.arange(N_RT))
rt_sims      = 1.0 - paired_cosine_distances(F_orig_rt, F_rt)
rt_root      = 1.0 - paired_cosine_distances(
    tree_root0.root.img_factors[rt_sub], projected_rt.img_factors)

print(f'Round-trip (full):  mean={rt_sims.mean():.4f}  std={rt_sims.std():.4f}')
print(f'Round-trip (root):  mean={rt_root.mean():.4f}  std={rt_root.std():.4f}')
print()
print('Active-sample fractions per node (first 8):')
for tn in tree_nodes0[:8]:
    sw = tn['stimulus_weights']
    print(f'  layer={tn["layer_idx"]}  path={tn["path"]}  active={(sw > 0.01).mean():.3f}')


### 4b — ID sanity check: val split-A vs split-B cross-similarity

In [ ]:
# ── ID sanity check: val split-A vs split-B cross-similarity (4 focus categories) ─
# Randomly partition each category's val samples into two halves (A / B).
# Half-A uses the original tree projection; half-B is re-projected via NNLS.
# Strong within-category / weak between-category similarity validates the BFT encoding.
rng_id   = np.random.default_rng(42)
SHOW_CI  = list(range(4))   # first 4 categories: airplane, ship, car, bicycle
N_BLK    = 40
blocks   = {}

for ci in SHOW_CI:
    ci_idx = np.where(all_targets0 == ci)[0].copy()
    rng_id.shuffle(ci_idx)
    half  = len(ci_idx) // 2
    idx_A = ci_idx[:half]
    idx_B = ci_idx[half:]
    if len(idx_A) > N_BLK: idx_A = rng_id.choice(idx_A, N_BLK, replace=False)
    if len(idx_B) > N_BLK: idx_B = rng_id.choice(idx_B, N_BLK, replace=False)
    li_B   = [l[idx_B] for l in layer_inputs0]
    proj_B = project_stimuli_onto_tree(tree_root0, li_B)
    if len(idx_A):
        blocks[f'A-{CATEGORY_NAMES[ci]}'] = extract_fingerprint_matrix(tree_root0, idx_A)
    if len(idx_B):
        blocks[f'B-{CATEGORY_NAMES[ci]}'] = extract_fingerprint_matrix(proj_B, np.arange(len(idx_B)))

F_cross  = np.concatenate(list(blocks.values()), axis=0)
bl_sizes = [len(v) for v in blocks.values()]
bl_ends  = list(np.cumsum(bl_sizes))
S_cross  = compute_stimulus_similarity(F_cross)
centres  = np.array([0] + bl_ends[:-1]) + np.array(bl_sizes) / 2

fig = plot_similarity_heatmap(S_cross, bl_sizes, list(blocks),
                               title='ID Sanity Check: val split-A vs split-B cross-similarity (4 focus categories)')
plt.savefig(os.path.join(FIG_DIR, 'id_cross_similarity.pdf'), bbox_inches='tight')
plt.show()

### 4c — Far-OOD: synthetic images

In [ ]:
# ── Far-OOD: 4 synthetic 3-channel image types ────────────────────────────────
N_FAR = 200; rng_f = np.random.default_rng(99); C = 3
_mn   = np.array(IMAGENET_MEAN)[:, None, None]
_st   = np.array(IMAGENET_STD)[:, None, None]
_chk  = (np.indices((IMG_SIZE, IMG_SIZE)).sum(0) % 2)[None].astype(np.float32)

_noise_raw = np.clip(rng_f.normal(0.5, 0.25,
                     (N_FAR, C, IMG_SIZE, IMG_SIZE)).astype(np.float32), 0, 1)
_orig_px   = all_images0[:N_FAR] * _st + _mn
_inv_norm  = (np.clip(1.0 - _orig_px, 0, 1) - _mn) / _st

far_ood_arrays = {
    'gaussian_noise': ((_noise_raw - _mn) / _st).astype(np.float32),
    'uniform_gray':   np.zeros((N_FAR, C, IMG_SIZE, IMG_SIZE), dtype=np.float32),
    'checkerboard':   np.broadcast_to(_chk, (N_FAR, C, IMG_SIZE, IMG_SIZE)).copy().astype(np.float32),
    'inverted_test':  _inv_norm.astype(np.float32),
}

far_ood_data = {}
for name, imgs in far_ood_arrays.items():
    ds     = TensorDataset(torch.from_numpy(imgs), torch.zeros(len(imgs), dtype=torch.long))
    loader = DataLoader(ds, 128, shuffle=False)
    raw    = collect_layer_dicts(model, loader, DEVICE,
                                  only_correct=False,
                                  layer_filter=squeezenet_spine_filter)
    d = dict(raw)
    d['layer_inputs']   = [ld['input_fmap'] for ld in raw['layer_data']]
    d['projected_root'] = project_onto_bft(tree_root0, model, loader,
                                            only_correct=False, device=DEVICE,
                                            layer_filter=squeezenet_spine_filter)
    d['factor_nodes']   = extract_factor_tree_nodes(d['projected_root'])
    far_ood_data[name]  = d
    print(f'{name:20s}  n={len(imgs)}')

n_ex  = 6
fig, axes = plt.subplots(len(far_ood_arrays), n_ex,
                          figsize=(n_ex * 2, len(far_ood_arrays) * 2.2))
for row, (name, imgs) in enumerate(far_ood_arrays.items()):
    for col in range(n_ex):
        raw = np.clip(imgs[col] * _st + _mn, 0, 1).transpose(1, 2, 0)
        axes[row, col].imshow(raw); axes[row, col].axis('off')
    axes[row, 0].set_ylabel(name, fontsize=9, rotation=30, ha='right', va='center')
plt.suptitle('Far OOD — example images', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'far_ood_examples.pdf'), bbox_inches='tight')
plt.show()

n_types = len(far_ood_data)
fig, axes = plt.subplots(1, n_types, figsize=(6 * n_types, 4.5))
for ax, (name, d) in zip(axes, far_ood_data.items()):
    all_idx = np.arange(len(d['images']))
    acts    = compute_factor_activations(d['factor_nodes'], all_idx)
    plot_factor_tree(d['factor_nodes'], acts, ax=ax, title=name)
plt.suptitle('Far OOD — factor tree activations', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'far_ood_trees.pdf'), bbox_inches='tight')
plt.show()

### 4d — Fingerprint embeddings and intra/inter-class similarity

In [ ]:
# ── Plot 7: embedding comparison (PCA fingerprints | PCA last-layer | PCA all-layers | MDS fingerprints) ──
N_EACH  = 60
rng_m   = np.random.default_rng(7)

def _pool_fmap(a, n):
    return a.mean(axis=(2, 3)) if a.ndim == 4 else a.reshape(n, -1)

F_parts, la_parts, full_la_parts, emb_labels, emb_conditions = [], [], [], [], []

# ID: focus val categories
id_sub = rng_m.choice(n_samples0, min(N_EACH, n_samples0), replace=False)
F_parts.append(extract_fingerprint_matrix(tree_root0, id_sub))
la_parts.append(layer_inputs0[-1][id_sub].reshape(len(id_sub), -1))
full_la_parts.append(np.concatenate([_pool_fmap(layer_inputs0[i][id_sub], len(id_sub))
                                      for i in range(len(layer_inputs0))], axis=1))
emb_labels    += list(all_targets0[id_sub])
emb_conditions += ['ID'] * len(id_sub)

# Far-OOD
ood_names = list(far_ood_data.keys())
for oi, (name, d) in enumerate(far_ood_data.items()):
    sub = rng_m.choice(len(d['images']), min(N_EACH, len(d['images'])), replace=False)
    F_parts.append(extract_fingerprint_matrix(d['projected_root'], sub))
    la_parts.append(d['layer_inputs'][-1][sub].reshape(len(sub), -1))
    full_la_parts.append(np.concatenate([_pool_fmap(d['layer_inputs'][i][sub], len(sub))
                                          for i in range(len(d['layer_inputs']))], axis=1))
    emb_labels    += [N_FOCUS + oi] * len(sub)
    emb_conditions += [name] * len(sub)

F_joint    = np.concatenate(F_parts,  axis=0)
la_joint   = np.concatenate(la_parts, axis=0)
emb_labels = np.array(emb_labels)

# Build full class_names dict: 0–7 = focus categories, 8+ = OOD condition names
full_class_names = dict(CAT_CLASS_NAMES)
for oi, name in enumerate(ood_names):
    full_class_names[N_FOCUS + oi] = name

fig7 = plot_embedding_comparison(
    F_joint, la_joint, emb_labels, full_class_names,
    condition_labels=emb_conditions,
    far_ood_conditions=ood_names,
    activations_all=np.concatenate(full_la_parts, axis=0),
    title='ID ImageNet focus classes vs Far-OOD',
)
fig7.savefig(os.path.join(FIG_DIR, 'embedding_comparison.pdf'), bbox_inches='tight')
plt.show(); plt.close(fig7)

# Fingerprint intra vs inter-class similarity (ID focus classes only)
F_all = extract_fingerprint_matrix(tree_root0, np.arange(n_samples0))
S_all = compute_stimulus_similarity(F_all)
intra_vals, inter_vals = [], []
for fi in range(N_FOCUS):
    mask = all_targets0 == fi
    intra = S_all[np.ix_(mask, mask)]
    intra_vals.extend(intra[np.triu_indices_from(intra, k=1)])
    for fj in range(fi + 1, N_FOCUS):
        inter_vals.extend(S_all[np.ix_(mask, all_targets0 == fj)].ravel())

intra_arr = np.array(intra_vals); inter_arr = np.array(inter_vals)
print(f'Intra-class similarity: {intra_arr.mean():.3f} ± {intra_arr.std():.3f}')
print(f'Inter-class similarity: {inter_arr.mean():.3f} ± {inter_arr.std():.3f}')

fig_hist, ax_hist = plt.subplots(figsize=(6, 4))
ax_hist.hist(intra_arr, bins=60, alpha=0.6, label='Intra-class', density=True)
ax_hist.hist(inter_arr, bins=60, alpha=0.6, label='Inter-class', density=True)
ax_hist.axvline(intra_arr.mean(), color='C0', ls='--')
ax_hist.axvline(inter_arr.mean(), color='C1', ls='--')
ax_hist.set(xlabel='Cosine similarity', ylabel='Density',
            title='Factor fingerprint: intra vs inter-class similarity (focus classes)')
ax_hist.legend()
fig_hist.tight_layout()
fig_hist.savefig(os.path.join(FIG_DIR, 'fingerprint_intra_inter.pdf'), bbox_inches='tight')
plt.show(); plt.close(fig_hist)

In [ ]:
# ── Plot 7b: Same 4-panel embedding, ID val data only ────────────────────────
F_id   = extract_fingerprint_matrix(tree_root0, np.arange(n_samples0))
act_id = layer_inputs0[-1].reshape(n_samples0, -1)
act_id_all = np.concatenate([_pool_fmap(layer_inputs0[i], n_samples0)
                               for i in range(len(layer_inputs0))], axis=1)

fig_id = plot_embedding_comparison(
    F_id, act_id, all_targets0, CAT_CLASS_NAMES,
    activations_all=act_id_all,
    title='ID val data — BFT fingerprint embeddings',
)
fig_id.savefig(os.path.join(FIG_DIR, 'embedding_id_only.pdf'), bbox_inches='tight')
plt.show(); plt.close(fig_id)

## §5 — Fingerprint figures (main paper & appendix)

### 5a — Main-paper figure

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# EXPORT — figure data for the SqueezeNet / ImageNet fingerprints
# Requires: tree_root0, all_targets0, n_samples0  (§2), rt_sims (§4a),
#           blocks / S_cross (§4b), far_ood_data (§4c)
# Writes:   figures/figdata/nb05_fingerprints.npz  (+ .json)
# ═══════════════════════════════════════════════════════════════════════════════
from src import figdata, figexport
from src.paper_figures import unit

CLASSES = [int(c) for c in np.unique(all_targets0)]
FAR_LABEL = {'gaussian_noise': 'noise', 'uniform_gray': 'gray',
             'checkerboard': 'checker', 'inverted_test': 'inverted'}

F     = extract_fingerprint_matrix(tree_root0, np.arange(n_samples0))
F_far = {n: extract_fingerprint_matrix(d['projected_root'],
                                       np.arange(len(d['images'])))
         for n, d in far_ood_data.items()}

_dims, _q = [], [tree_root0.root]
while _q:
    _n = _q.pop(0)
    for _k in range(_n.img_factors.shape[1]):
        _dims.append((_n.layer_idx, _n.path[0] if _n.path else -1, _k))
    _q.extend(_n.children)
_dims = np.array(_dims)
_top  = _dims[:, 0].max()
BLOCKS = [[int(r)] + [i for i in range(len(_dims))
                      if _dims[i, 0] != _top and _dims[i, 1] == _dims[r, 2]]
          for r in np.where(_dims[:, 0] == _top)[0]]


def centroid_cos(Fm):
    U = unit(Fm)
    c = U.mean(0)
    return U @ (c / (np.linalg.norm(c) + 1e-12))


CENT = unit(np.stack([F[all_targets0 == c].mean(0) for c in CLASSES]))
COND = ([dict(label='ID val', values=centroid_cos(F), color_key='id_data')] +
        [dict(label=FAR_LABEL.get(n, n), values=centroid_cos(F_far[n]),
              color_key='far_ood') for n in far_ood_data])
LIKE = [dict(label=lab, values=(unit(X) @ CENT.T).max(1), color_key=ck)
        for lab, X, ck in ([('ID val', F, 'id_data')] +
                           [(FAR_LABEL.get(n, n), F_far[n], 'far_ood')
                            for n in far_ood_data])]

# ── Activation baseline for the fingerprint-vs-activations panels (Fig. 4h, 4i) ─
# The network's own representation on the SAME stimuli, in the same row order, read
# straight off the layer dicts collected in §2 -- so it is aligned with F by
# construction. scripts/add_activation_baselines.py cannot supply this one: it would
# need the ImageNet val images, which no bundle ships.
# Each spine layer's input feature map is global-average-pooled to one value per
# channel (the nb01-nb04 convention); features.0's input is the raw image, so it is
# dropped. reps[-1] is the representation the figures compare against by default.
_pool = lambda x: (x.reshape(len(x), x.shape[1], -1).mean(2) if x.ndim == 4 else x)
_acts = [(ld['name'], _pool(np.asarray(ld['input_fmap'], np.float32)))
         for ld in data0['layer_data'] if ld['name'] != 'features.0']
_ACT_ALL = np.concatenate([A for _, A in _acts], axis=1)
_ACT_CLS = next(A for n, A in _acts if n.startswith('classifier'))
print('per-layer activation dims:', [(n, int(A.shape[1])) for n, A in _acts])
print(f'all layers {_ACT_ALL.shape} ({_ACT_ALL.nbytes / 1e6:.1f} MB) | '
      f'classifier {_ACT_CLS.shape} ({_ACT_CLS.nbytes / 1e6:.1f} MB)')

ACT = dict(aligned=1, index=np.arange(n_samples0),
           labels=np.asarray(all_targets0),
           reps=[dict(label='all layers act.', X=_ACT_ALL.astype(np.float32),
                      dim=int(_ACT_ALL.shape[1])),
                 dict(label='classifier act.', X=_ACT_CLS.astype(np.float32),
                      dim=int(_ACT_CLS.shape[1]))])

D = figdata.save('nb05_fingerprints', dict(
    classes=CLASSES, n_classes=len(CLASSES),
    category_names=[str(c) for c in CATEGORY_NAMES],
    class_names=[str(CAT_CLASS_NAMES[c]) if c in CAT_CLASS_NAMES else str(c)
                 for c in CLASSES],
    n_factors=F.shape[1], dims=_dims,
    col_order=[c for b in BLOCKS for c in b],
    blk_edge=np.cumsum([len(b) for b in BLOCKS])[:-1],
    block_sizes=[len(b) for b in BLOCKS],
    fp_mean_by_class=np.stack([F[all_targets0 == c].mean(0) for c in CLASSES]),
    cond=COND, like=LIKE, rt_sims=rt_sims, act=ACT,
    split_cross=dict(matrix=np.asarray(S_cross, np.float32),
                     block_sizes=np.asarray(bl_sizes, int)),
    fp=dict(id=F.astype(np.float32), id_targets=np.asarray(all_targets0),
            far=[dict(label=FAR_LABEL.get(n, n), F=F_far[n][:200].astype(np.float32))
                 for n in far_ood_data])))
figdata.summary('nb05_fingerprints')


### 5b — Appendix figure

In [ ]:
# (appendix figure: build it in notebooks/fig05_imagenet.ipynb from the exported bundle)


## §6 — Hyperparameter check (held-out arbor R²)

Re-derives the per-layer circuit rank with the metric-free C0 rule (`src.hp_selection`), on the circuit tree's own arbors. Confirms the `K_MAX` in §1 sits at the reconstruction plateau; reads no fingerprint metric. Cached.

In [ ]:
from src import node_pos_arbor, nodes_per_layer, select_ranks, cached_result
_C = ANALYSIS_CTX
_npl = nodes_per_layer(_C['tree_circuit'], max_nodes=2)
_arbors = {li: [node_pos_arbor(nd, _C['layer_inputs'][li]) for nd in nds]
           for li, nds in _npl.items()}
hp_sel = cached_result(
    _C['tag'] + '_hpsel',
    lambda: select_ranks(_arbors, _C['labels_task'], k_cap=_C['k_cap'],
                         n_classes=_C['n_classes'], last_extra=_C['last_extra']),
    params=dict(kcap=_C['k_cap'], n=len(_C['labels_task']),
                kmax=list(_C['tree_circuit'].root.lambdas.shape)))
print('held-out K* per layer:', hp_sel['profile']['k_from_criterion'])
print('assembled profile     k_max=%s  n_branches=%s'
      % (hp_sel['profile']['k_max'], hp_sel['profile']['n_branches']))
print('§1 circuit k_max was :', _C.get('k_max_cfg'))

## §7 — Validation (faithfulness + class-relevant structure)

On the **circuit** tree: NMF init-stability per layer, causal-reconstruction fidelity (fc layers only), and the weight-term control (arbor-NMF vs activation-NMF separability). All via `src`; cached.

In [ ]:
from src import (compute_nmf_stability, summarize_validation, node_pos_arbor,
                 nodes_by_layer, weight_term_control, cached_result)
import numpy as _np
_C = ANALYSIS_CTX
def _run_validation():
    tc = _C['tree_circuit']
    nbl = nodes_by_layer(tc)
    stability = {}
    for li, nd in nbl.items():
        X = node_pos_arbor(nd, _C['layer_inputs'][li])
        k = int(nd.img_factors.shape[1])
        sim, _ = compute_nmf_stability(
            X if X.shape[0] <= 800 else X[_np.random.default_rng(0).choice(X.shape[0], 800, False)],
            k, n_seeds=_C.get('stab_seeds', 5), max_iter=300)
        stability[int(li)] = float(sim[~_np.eye(len(sim), dtype=bool)].mean())
    recon = summarize_validation(tc.nodes())        # None when no fc node was validated
    wtc = weight_term_control(tc, _C['labels_task'], _C['layer_inputs'])
    return {'stability': stability,
            'recon_overall': (recon['overall'] if recon else None),
            'weight_term': wtc}
val = cached_result(_C['tag'] + '_validation', _run_validation,
                    params=dict(n=len(_C['labels_task']), tag='circ'))
print('NMF stability per layer:', {k: round(v, 3) for k, v in val['stability'].items()})
if val['recon_overall']:
    print('causal recon preact_R2 (median):',
          round(val['recon_overall']['preact_r2']['median'], 3))
print('weight-term control: arbor NMF sil=%.3f  vs activation NMF sil=%.3f'
      % (val['weight_term']['arbor_nmf']['sil'], val['weight_term']['activation_nmf']['sil']))

## §8 — Causal pruning

Prunes each class circuit's weights in BFT-importance order on the **circuit** tree and measures target vs bystander accuracy (`src.pruning`, wrapping `ablation_sweep`). One seed here; add checkpoints for the full seed×class grid on the cluster. Cached.

In [ ]:
from src import run_pruning, cached_result
import numpy as _np
_C = ANALYSIS_CTX
if _C.get('skip_pruning'):
    print('pruning skipped for this model:', _C.get('skip_pruning'))
    prune = None
else:
    _reps = [{'seed': 0, 'model': _C['model'], 'tree': _C['tree_circuit'],
              'layer_names': _C.get('layer_names'), 'targets': _C['labels_task']}]
    _targets = (list(range(_C['n_classes'])) if _C['n_classes'] <= 10
                else list(range(_C['n_classes']))[:10])
    prune = cached_result(
        _C['tag'] + '_pruning',
        lambda: run_pruning(_reps, _C['eval_loader'], _targets, n_classes=_C['n_classes'],
                            fractions=_C.get('prune_fractions', (0.02, 0.05, 0.1, 0.2)),
                            label_transform=_C.get('label_transform'), device=_C.get('device'),
                            n_random_repeats=_C.get('n_random', 5), verbose=1)['aggregate'],
        params=dict(n=len(_targets), tag='prune'))
    for m in ('bft_top', 'bft_bottom', 'random'):
        if m in prune['methods']:
            print('%-11s target drop@0.2 (mean over classes): %+.3f'
                  % (m, _np.mean(prune['drops'][m]['target'])))

## §9 — Fingerprint separability (C1.8)

On the **fingerprint** tree: is the factor fingerprint more class-separable than the network's own activations, and where in the tree does that live? `src.separability` gives silhouette + kNN for the whole tree, its upper/lower slices, and the penultimate / full-activation baselines (native and dim-matched). Cached.

In [ ]:
from src import separability_evaluate, cached_result
_C = ANALYSIS_CTX
sep = cached_result(
    _C['tag'] + '_separability',
    lambda: separability_evaluate(_C['tree_fp'], _C['labels_fine'], _C['layer_inputs']),
    params=dict(n=len(_C['labels_fine']), tag='fp'))
_n = sep['native']
print('native silhouette:  fp_full=%.3f  output_only=%.3f  top_half=%.3f  spine=%.3f'
      % (_n['fp_full']['sil'], _n.get('fp_output_only', {}).get('sil', float('nan')),
         _n.get('fp_top_half', {}).get('sil', float('nan')), _n.get('fp_spine', {}).get('sil', float('nan'))))
print('activation baselines: penult=%.3f  full=%.3f'
      % (_n['act_penult']['sil'], _n['act_full']['sil']))
_p = sep['paired'].get('fp_full__vs__act_penult')
if _p:
    print('dim-matched @%d: fp(pca)=%.3f vs penult(pca)=%.3f'
          % (_p['match_dim'], _p['A_pca']['sil'], _p['B_pca']['sil']))